# Validation: origin-error metrics on all 5 benchmark incidents

This notebook replays the **transport (backward Lagrangian backtracking)** stage
for every benchmark incident in `data/benchmarks/incidents.json` and computes a
single, interpretable validation quantity: the **origin-error**.

## What "origin-error" measures

For each incident the pipeline:

1. Takes the known oil-slick / incident location `(lon_src, lat_src)`.
2. Runs the backward Lagrangian tracker (`engines/transport/lagrangian_tracker.py`)
   from that point for `duration_hours`, advecting 100 particles against the
   ERA5 wind + CMEMS current forcing stored in each incident's `final_metocean.nc`.
3. Records the backtracked **origin probability centroid** `(lon_hat, lat_hat)`.

> origin-error = geodesic distance between the backtracked centroid and the known
> source coordinate (km).

A small error means the backtracking faithfully reconstructs the source
(consistent with a fresh, weakly-advected spill). A large error flags either
strong advection that the forcing does not explain, or a **metocean data fault**
(e.g. a file with the wrong date range).

## What this notebook needs from you
- `GFW_API_KEY` in `.env` for the AIS / attribution replay (optional, used in
  the last section; the transport-based origin-error does **not** need it).
- Confirmation that the `coordinates` in `incidents.json` are the true
  **source** locations (they are currently the slick / incident coordinates).
- For the optional SAR detection replay: a valid `CDSE_USERNAME`/`PASSWORD`
  (already present in `.env`) and time to download a ~1.7 GB GRD product per
  incident (Sentinel-1 is only available for post-2014 incidents).


In [ ]:
import sys, json
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

incidents = json.load(open(PROJECT_ROOT / 'data' / 'benchmarks' / 'incidents.json'))
print(f'Loaded {len(incidents)} benchmark incidents')
for i in incidents:
    print(f"  {i['id']:40s} {i['coordinates']}  {i['date']}")


In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    """Great-circle distance in km between two (lon,lat) points."""
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

# sanity check: ~111 km per degree latitude
print('1 deg lat ~', round(haversine_km(0,0,0,1),1), 'km')


## Metocean data availability

Each incident needs a `final_metocean.nc` with `uo/vo` (currents) and ideally
`u10/v10` (wind). Map incident -> forcing file.


In [ ]:
import xarray as xr

INCIDENT_FILES = {
    'msc_chitra_khalijia3_mumbai_2010': 'data/processed/metocean/msc_chitra_khalijia3_mumbai_2010/final_metocean.nc',
    'mt_jipro_neftis_mumbai_2018':     'data/processed/metocean/mt_jipro_neftis/final_metocean.nc',
    'gal_constructor_mumbai_2021':     'data/processed/metocean/mumbai/final_metocean.nc',
    'ennore_chennai_coastal_2017':     'data/processed/metocean/ennore_chennai_coastal_2017/final_metocean.nc',
    'kandla_gulf_kutch_2023':          'data/processed/metocean/kandla_gulf_kutch_2023/final_metocean.nc',
}

def file_time_range(path):
    if not Path(path).exists():
        return None
    ds = xr.open_dataset(path)
    tname = 'time' if 'time' in ds.coords else ('valid_time' if 'valid_time' in ds.coords else None)
    if tname is None:
        ds.close(); return None
    t = ds[tname]
    tr = (str(t.values.min()), str(t.values.max()))
    has = {v: v in ds.data_vars for v in ['uo','vo','u10','v10']}
    ds.close()
    return tr, has, dict(ds.sizes)

print('%-42s %-26s %s' % ('incident','time range','has uo/vo/u10/v10'))
for kid in INCIDENT_FILES:
    info = file_time_range(INCIDENT_FILES[kid])
    if info is None:
        print(f"{kid:42s} MISSING")
    else:
        tr, has, _ = info
        flags = ''.join('1' if has[k] else '0' for k in ['uo','vo','u10','v10'])
        print(f'{kid:42s} {tr[0]} -> {tr[1]}    {flags}')


> **Note on GAL Constructor 2021:** `mumbai/final_metocean.nc` previously held
> a 2007-08..2010-08 file (wrong period). It was regenerated via
> `prepare_incident_metocean.py --incident-id gal_constructor_mumbai_2021
> --start-date 2021-05-14 --end-date 2021-05-21 --bbox 18.2 71.8 19.5 73.2
> --output data/processed/meteocean/mumbai/final_metocean.nc`, fetching fresh
> ERA5 (cyclone window, 6-hourly) + CMEMS currents. It now spans
> **2021-05-14..2021-05-21** (32 steps) with u10/v10/uo/vo, so the replay below
> resolves for GAL. Expect a larger origin-error here than the floating-slick
> incidents: GAL Constructor was **grounded during Cyclone Tauktae**, so the
> vessel was not advecting with the current; backtracking a spreading sheen
> from a stationary source under cyclone-force winds spreads the origin out.


## Replay: backward Lagrangian backtracking per incident

We start particles at the incident coordinates and track them **backward** in
time for a configurable number of hours. The origin-error is then the geodesic
distance from the backtracked centroid back to the source coordinate.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from engines.transport.lagrangian_tracker import LagrangianTracker

DURATION_HOURS = 24
N_PARTICLES = 100   # ~20s per incident on CPU; raise for a smoother PMF
SEED = 42

results = []
for inc in incidents:
    kid = inc['id']
    lon_src, lat_src = inc['coordinates']
    inc_date = inc['date']
    path = INCIDENT_FILES.get(kid)
    row = {'incident': kid, 'name': inc['incident_name'],
           'source_lon': lon_src, 'source_lat': lat_src,
           'date': inc_date, 'status': 'OK'}

    if path is None or not Path(path).exists():
        row['status'] = 'NO_FORCING'
        row['origin_error_km'] = np.nan
        results.append(row)
        continue

    # date sanity check: does the forcing cover the incident date?
    tr, has, _ = file_time_range(path)
    forcing_covers = False
    if tr is not None:
        t0 = np.datetime64(inc_date)
        forcing_covers = (t0 >= np.datetime64(tr[0][:10])) and (t0 <= np.datetime64(tr[1][:10]))

    if not forcing_covers:
        row['status'] = 'NO_FORCING (date mismatch)'
        row['origin_error_km'] = np.nan
        results.append(row)
        continue

    try:
        np.random.seed(SEED)
        tracker = LagrangianTracker(path)
        # detection time: 00:00 UTC (local = UTC+5:30)
        start_time = f"{inc_date}T00:00:00"
        particles = tracker.track_backward(lon_src, lat_src, start_time,
                                           DURATION_HOURS,
                                           num_particles=N_PARTICLES)
        origin = tracker.compute_origin_probability(particles)
        lon_hat, lat_hat = origin['centroid']
        err_km = haversine_km(lon_src, lat_src, lon_hat, lat_hat)
        row['n_active'] = len(particles)
        row['origin_lon'] = round(lon_hat, 4)
        row['origin_lat'] = round(lat_hat, 4)
        row['origin_error_km'] = round(err_km, 2)
        row['std_lon'] = round(origin['std_dev'][0], 4)
        row['std_lat'] = round(origin['std_dev'][1], 4)
    except Exception as e:
        row['status'] = f'ERROR: {type(e).__name__}: {e}'
        row['origin_error_km'] = np.nan
    results.append(row)

import pandas as pd
df = pd.DataFrame(results)
df


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ok = df[df['origin_error_km'].notna()].copy()
print('Backtracking origin-error (km), mean of active particles')
print(df[['incident','status','n_active','origin_error_km','origin_lon','origin_lat']].to_string(index=False))
print()
if len(ok):
    errs = ok['origin_error_km']
    print('--- aggregate ---')
    print(f'  mean   : {errs.mean():.2f} km')
    print(f'  median : {errs.median():.2f} km')
    print(f'  rmse   : {np.sqrt((errs**2).mean()):.2f} km')
    print(f'  max    : {errs.max():.2f} km')


In [ ]:
# Per-incident bar chart of origin-error
ok = df[df['origin_error_km'].notna()]
err_map = dict(zip(ok['incident'], ok['origin_error_km']))
all_ids = [i['id'] for i in incidents]
vals = [err_map.get(k, np.nan) for k in all_ids]

fig, ax = plt.subplots(figsize=(10,4))
colors = ['#2e7d32' if not np.isnan(v) else '#c62828' for v in vals]
ax.bar(all_ids, vals, color=colors)
ax.axhline(ok['origin_error_km'].mean() if len(ok) else 0, color='k', ls='--', label='mean')
ax.set_ylabel('origin-error (km)')
ax.set_title('Backward-tracked origin error vs known source (lower = better)')
for tick in ax.get_xticklabels():
    tick.set_rotation(35); tick.set_ha('right')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Map: source vs backtracked origin
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    has_cartopy = True
except Exception:
    has_cartopy = False

if has_cartopy:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(1,1,1, projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.4)
    ax.set_extent([66, 82, 6, 24], crs=ccrs.PlateCarree())
    for row in df.itertuples():
        if np.isnan(row.origin_error_km):
            continue
        ax.plot(row.source_lon, row.source_lat, 'go', transform=ccrs.PlateCarree(), ms=8)
        ax.plot(row.origin_lon, row.origin_lat, 'ro', transform=ccrs.PlateCarree(), ms=6)
        ax.annotate(row.incident[:18], (row.source_lon, row.source_lat),
                    transform=ccrs.PlateCarree(), fontsize=7)
    plt.title('Green = known source, Red = backtracked origin')
    plt.show()
else:
    print('cartopy not installed - skipping map (pip install cartopy to enable)')


## Optional: SAR-detection origin error (post-2014 incidents)

For incidents with a Sentinel-1 scene we can also measure how close the CFAR
dark-spot detection centroid is to the known source. This downloads a ~1.7 GB
GRD product per incident, so run it deliberately. If a product was already
downloaded into `data/raw/sar/<incident>/`, pass `safedir` to skip the download.


In [ ]:
# Leave uncommented to actually run; requires CDSE creds (in .env).
# To re-run a specific already-downloaded product, set SAFEDIR.
import os
RUN_SAR = False   # <-- set True to trigger live downloads
SAFEDIR = None    # e.g. 'data/raw/sar/mt_jipro_neftis_mumbai_2018/download/extracted/s1_00d06b1e'

def sar_origin_error(incident_id, safedir=None):
    from engines.detection.sar_detector import SARDetector
    det = SARDetector.__new__(SARDetector)
    detfile = 'data/raw/sar/%s/detections.json' % incident_id
    if os.path.exists(detfile):
        js = json.load(open(detfile))
        dets = js.get('detections', [])
        if dets:
            cx = np.mean([d['centroid_geo'][0] for d in dets])
            cy = np.mean([d['centroid_geo'][1] for d in dets])
            return cx, cy, js.get('product', {}).get('name'), dets
    return None

if RUN_SAR:
    print('SAR replay requires live CDSE downloads - running only for incidents with saved detections.json')
    for inc in incidents:
        res = sar_origin_error(inc['id'])
        if res is not None:
            cx, cy, prod, dets = res
            lon_s, lat_s = inc['coordinates']
            err = haversine_km(lon_s, lat_s, cx, cy)
            print(f"{inc['id']:40s} SAR-centroid err={err:7.1f} km  n_det={len(dets)} product={prod}")
        else:
            print(f"{inc['id']:40s} no saved detection")
else:
    print('RUN_SAR=False (skipped). Saved detections found:')
    for inc in incidents:
        res = sar_origin_error(inc['id'])
        if res:
            cx, cy, prod, dets = res
            lon_s, lat_s = inc['coordinates']
            err = haversine_km(lon_s, lat_s, cx, cy)
            print(f"  {inc['id']:40s} err={err:7.1f} km n_det={len(dets)} {prod}")
        else:
            print(f"  {inc['id']:40s} (no detection json)")


## Optional: AIS + attribution replay

Checks whether the true source vessel is ranked first. Requires `GFW_API_KEY`
in `.env`. The transport origin centroid from above is used as the search seed.


In [ ]:
def replay_attribution(inc, origin_centroid, duration_hours):
    from engines.pipeline import _run_ais, _run_attribution
    # build detection time in the GFW-expected ISO-8601 format
    detection_time = f"{inc['date']}T00:00:00"
    suspects, available = _run_ais(
        {'bbox': [origin_centroid[0]-0.1, origin_centroid[1]-0.1,
                  origin_centroid[0]+0.1, origin_centroid[1]+0.1]},
        detection_time, duration_hours)
    if not suspects:
        return None, available
    ranked = _run_attribution(suspects, origin_centroid)
    return ranked, available

RUN_AIS = False  # requires GFW_API_KEY
if RUN_AIS:
    for row in df.itertuples():
        if np.isnan(row.origin_error_km):
            print(f"{row.incident:40s} skip (no forcing)")
            continue
        ranked, avail = replay_attribution(
            {'date': row.date}, [row.origin_lon, row.origin_lat], DURATION_HOURS)
        if not avail:
            print(f"{row.incident:40s} GFW unavailable")
            continue
        if not ranked:
            print(f"{row.incident:40s} no vessels in origin window")
            continue
        top = ranked[0]
        print(f"{row.incident:40s} top: {top['vessel_name']} score={top['attribution_score']} (of {len(ranked)})")
else:
    print('RUN_AIS=False. Set RUN_AIS=True and ensure GFW_API_KEY is in .env to rank vessels.')


## Summary / interpretation

The **origin-error** table makes the transport stage directly auditable:

- `OK` + small error ⇒ backward backtracking faithfully reconstructs the source.
- `OK` + large error ⇒ strong advection not explained by the forcing, or a bad
  detection seed (GAL Constructor, grounded during Cyclone Tauktae, is an
  expected case for a larger error).
- `NO_FORCING` / `date mismatch` ⇒ a data problem surfaced by validation. Fix
  the forcing file, not the model.

To close out the validation you need to provide from your side:

1. **Ground-truth source coordinates** for each incident (or confirmation that
   `incidents.json` `coordinates` are the true spill source).
2. **GFW_API_KEY** in `.env` to run the AIS/attribution replay.
3. **CDSE time budget** (~1.7 GB/product) if you want the live SAR detection
   replay across all post-2014 incidents (Jipro's detection is already saved).
